# PhasePhyto Run Overview

Loads one saved run from Drive, prints the manifest, artifact paths, and available runs. Use this first when deciding which run to inspect.


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    # Set this to a specific run folder name under drive_project_dir/runs.
    # If None, the latest run by folder name is used.
    "run_name": None,
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")


In [ ]:
# ============================================================
# MOUNT DRIVE + LOCATE RUN
# ============================================================

from pathlib import Path
import json

from google.colab import drive
from IPython.display import Image, Markdown, display

drive.mount("/content/drive", force_remount=False)

PROJECT_DIR = Path(CONFIG["drive_project_dir"])
RUNS_DIR = PROJECT_DIR / "runs"

if not RUNS_DIR.exists():
    raise RuntimeError(f"No runs directory found: {RUNS_DIR}")

if CONFIG["run_name"]:
    RUN_DIR = RUNS_DIR / CONFIG["run_name"]
else:
    candidates = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()])
    if not candidates:
        raise RuntimeError(f"No run directories found under {RUNS_DIR}")
    RUN_DIR = candidates[-1]

MANIFEST_PATH = RUN_DIR / "run_manifest.json"
if not MANIFEST_PATH.exists():
    raise RuntimeError(f"No run_manifest.json found at {MANIFEST_PATH}")

manifest = json.loads(MANIFEST_PATH.read_text())
artifacts = manifest.get("artifacts", {})

print(f"Loaded run: {RUN_DIR}")
print(f"Manifest: {MANIFEST_PATH}")


In [ ]:
# ============================================================
# SHOW MANIFEST + ARTIFACT PATHS
# ============================================================

print(json.dumps(manifest, indent=2))

print("\nArtifact paths:")
for name, value in artifacts.items():
    path = Path(value) if value and not any(ch in str(value) for ch in "*") else value
    exists = path.exists() if isinstance(path, Path) else "glob"
    print(f"  {name}: {value} | exists={exists}")


In [ ]:
# ============================================================
# LIST AVAILABLE RUNS
# ============================================================

print("Available runs:")
for run_dir in sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()]):
    marker = "<-- current" if run_dir == RUN_DIR else ""
    print(f"  {run_dir.name} {marker}")
